# Figure 1 -- the evaluation regions

Standard deviation of the band-passed sea surface height over the record, one panel
per region, with a black box marking the strip used for the wavenumber--frequency
spectra of Figure S6.

The field is derived from the mapped altimetry; each file carries the credit its
licence requires in its `licence` attribute, reproduced below the figure.

Panel widths follow each domain's aspect ratio so the three frames share a height.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    PROJ, HAVE_CARTOPY = ccrs.PlateCarree(), True
except ImportError:
    PROJ, HAVE_CARTOPY = None, False
    print("cartopy is not installed -- drawing the maps without coastlines")

# Cartopy fetches its coastline data from the network the first time it is used.
# Try once here, so a machine that is offline or has no cached copy degrades to a
# map without coastlines instead of failing in the middle of a figure.
HAVE_COAST = False
if HAVE_CARTOPY:
    try:
        next(iter(cfeature.COASTLINE.geometries()))
        HAVE_COAST = True
    except Exception as exc:
        print(f"cartopy cannot reach its coastline data ({type(exc).__name__}) -- "
              f"drawing the maps without coastlines")

TKW = dict(transform=PROJ) if HAVE_CARTOPY else {}


def map_features(ax, labels=True, land="lightgray", lw=0.4, zland=3, zcoast=4):
    """Coastlines and gridlines where available; plain axes otherwise."""
    if HAVE_COAST:
        ax.add_feature(cfeature.LAND, facecolor=land, zorder=zland)
        ax.add_feature(cfeature.COASTLINE, linewidth=lw, zorder=zcoast)
    if HAVE_CARTOPY:
        gl = ax.gridlines(draw_labels=labels, linestyle="--", lw=0.2)
        gl.top_labels = gl.right_labels = False
        return gl
    ax.tick_params(labelsize=7)
    return None


import matplotlib.patches as mpatches

files = by_region(need("Figure1_ssh_variability_*.nc"))
PANEL_H, CMAP = 3.4, "turbo"

data, aspects = [], []
for region, path in files:
    with xr.open_dataset(path) as ds:
        data.append((region, ds.load()))
        aspects.append((ds.attrs["box_lon_max"] - ds.attrs["box_lon_min"]) /
                       (ds.attrs["box_lat_max"] - ds.attrs["box_lat_min"]))

fig, axes = plt.subplots(1, len(data), dpi=130,
                         figsize=(sum(a * PANEL_H for a in aspects) + 0.8,
                                  PANEL_H + 1.5),
                         subplot_kw={"projection": PROJ} if HAVE_CARTOPY else None,
                         gridspec_kw={"width_ratios": aspects})
axes = np.atleast_1d(axes)

for k, (ax, (region, ds), a) in enumerate(zip(axes, data, aspects)):
    lon = ds.lon.values.copy()
    if ds.attrs["box_lon_min"] < 0:
        lon = np.where(lon > 180, lon - 360.0, lon)
    extent = [ds.attrs["box_lon_min"], ds.attrs["box_lon_max"],
              ds.attrs["box_lat_min"], ds.attrs["box_lat_max"]]
    if HAVE_CARTOPY:
        ax.set_extent(extent, crs=PROJ)
    else:
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
    pcm = ax.pcolormesh(lon, ds.lat, ds.ssh_std, cmap=CMAP, vmin=0,
                        vmax=ds.attrs["colour_limit_cm"], shading="auto",
                        **TKW)
    x0, x1 = ds.attrs["spectral_box_lon_min"], ds.attrs["spectral_box_lon_max"]
    y0, y1 = ds.attrs["spectral_box_lat_min"], ds.attrs["spectral_box_lat_max"]
    ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                    facecolor="none", edgecolor="k", lw=1.8,
                                    zorder=5, **TKW))
    gl = map_features(ax, lw=0.5)
    if gl is not None:
        gl.xlabel_style = gl.ylabel_style = {"size": 9}
    ax.set_title(f"({chr(97 + k)}) {region}", fontsize=13, fontweight="bold",
                 loc="left")
    cb = fig.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.07,
                      shrink=0.5 * aspects[0] / a, aspect=20)
    cb.set_label("SSH standard deviation (cm)", fontsize=9)
    cb.ax.tick_params(labelsize=8)

fig.subplots_adjust(left=0.04, right=0.99, top=0.92, bottom=0.14, wspace=0.06)
out = FIGDIR / "Figure1_regions.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("wrote", out)
print("\n" + data[0][1].attrs["licence"])
for _, ds in data:
    ds.close()
